In [1]:
import pandas as pd
import os

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import TavilySearchTool, ScrapeWebsiteTool
from pydantic import BaseModel, Field
from typing import List, Optional

In [2]:
from dotenv import load_dotenv
load_dotenv("../.env.local", override=True)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPEN_AI_API_KEY")

In [3]:
search_tool = TavilySearchTool(api_key=TAVILY_API_KEY)
scrape_tool = ScrapeWebsiteTool()  
llm = LLM(model="gpt-4o", api_key=OPENAI_API_KEY)

In [39]:
class ReleaseCandidate(BaseModel):
    product_name: str = Field(..., description="Name of the product")
    brand: Optional[str] = Field(None, description="Brand if known")
    release_date: str = Field(..., description="Date product releases")
    retail_price: Optional[int] = Field(..., description="Expected retail price of product")
    retailers: Optional[str] = Field(..., description="Retailers confirmed to be selling the item")
    seed_sources: List[str] = Field(..., description="URLs confirming the release")

class ScoutOutput(BaseModel):
    candidates: List[ReleaseCandidate]

class ReleaseItems(ReleaseCandidate):
    resale_estimate: int = Field(..., description="Estimated resale value")
    confidence_score: float = Field(..., description="Level of confidence from 0-100 that resale_estimate is correct")

class AnalystOutput(BaseModel):
    items: List[ReleaseItems]

## Agents

In [40]:
sneaker_scout = Agent(
    role="Upcoming Sneaker Release Scout",
    goal="""Identify upcoming sneaker releases.""",
    backstory="""
    You are a master web scraper who is highly resourceful and can easily navigate the internet to find relevant information and 
    Extract it in easily ingestible formats. You do not get distracted by irrelevant articles/information.
    """,
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

sneaker_market_analyst = Agent(
    role="Sneaker Resell Market Analyst",
    goal="Accurately project resale values for upcoming sneaker releases",
    backstory="""You are a long-time sneaker reseller and hypebeast. You have an expert understanding of how cultural trends,
    historical performance and market factors impact the potential profitability of sneakers on the secondary market.""",
    tools=[search_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


## Tasks

In [42]:
sneaker_scout_task = Task(
    description="""
    Compile information on upcoming sneaker releases by scraping information from reputable release calendars on
    sites such as https://www.sneakerfiles.com/release-dates/, nicekicks.com, sneakernews.com, and goat.com. Find and navigate to the release calendar pages on each site
    to find organized information on upcoming releases.
    
    - Today is {today}. Only include releases between {today} and {cutoff_date}.

    Deliverable: Return up to 20 releases ordered by the soonest upcoming release.
    """,
    expected_output="""ScoutOutput with exactly 20 upcoming sneaker releases in the date window, each with 1–2 sources. You're outputted
    items should closely match the items on the release calendars of the sites.""",
    output_pydantic=ScoutOutput,
    agent=sneaker_scout,
)

sneaker_market_analyst_task = Task(
    description="""
    Given a list of upcoming sneaker releases, do research and analyze each item one by one to come up with a resell price prediction
    and confidence score for that prediction.
    Consider historical trends and the performance of similar sneakers (same model or release type (collaboration, limited release etc.)
    Use StockX sale price as the most reliable indicator for fair resale value.
    Based on your research, make a resell price prediction. If you are considering a wide range of values, choose the 50th percentile value
    and lower your confidence score.
    If you can't find similar items, or there is no history of consistent price trends for similar items, lower your confidence score.

    Important note: Using StockX sale value for the exact item you are analyzing is not reliable because the item has not released to 
    the public yet and pre-release prices are always inflated.
    Instead, look at prices for previously released items of the same model or line as the sneaker you are analyzing. Use your intuition
    about what makes items popular, to evaluate unique items such as collaborations, and limited releases.

    Deliverable: For each item in the given list, generate a resale price prediction of what you think the item will sell for on the
    secondary resell market within one month of purchase and a confidence score based on how confident you are in that prediction.
    You will output AnalystOutput with predictions and confidence scores for all items provided in the original list.
    """,
    expected_output="""
    AnalystOutput with the same number of items as the original given list, including resale price predictions and confidence scores
    for each item.
    """,
    output_pydantic=AnalystOutput,
    agent=sneaker_market_analyst,
    context=[sneaker_scout_task],
)

In [43]:
from datetime import date, timedelta

today = date.today()
cutoff = today + timedelta(days=21)
window_month = date.today()

test_crew = Crew(
    agents=[sneaker_scout, sneaker_market_analyst],
    tasks=[sneaker_scout_task, sneaker_market_analyst_task],
    verbose=True,
    process=Process.sequential,
)

result = test_crew.kickoff(
    inputs={
        "today": today.isoformat(),
        "cutoff_date": cutoff.isoformat(),
    }
)

result

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 42bb6a29-3456-4f5f-b34d-97e9b66991ac                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Upcoming Sneaker Release Scout                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Compile information on upcoming sneaker releases by scraping information from reputable release calendars  │
│  on                                                                                                             │
│      sites such as https://www.sneakerfiles.com/release-dates/, nicekicks.com, sneakernews.com, and goat.com.   │
│  Find and navigate to the release calendar pages on each site                                                   │
│      to find organized information on upcoming releases.                                                        │
│                                                                                                                 │
│      - Today is 2025-12-30. Only include releases between 2025-12-30 and 2026-01-20.                            │
│                                                                                                                 │
│      Deliverable: Return up to 20 releases ordered by the soonest upcoming release.                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/caedinm/Projects/flipper/venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


/Users/caedinm/Projects/flipper/venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Upcoming Sneaker Release Scout                                                                          │
│                                                                                                                 │
│  Thought: To gather information on upcoming sneaker releases, I will initiate searches on reliable sneaker      │
│  websites to locate the release calendar pages. The target period is from 2025-12-30 to 2026-01-20.             │
│  First, I will locate and gather information from sneakerfiles.com, followed by other sources.                  │
│                                                                                                                 │
│  Using Tool: Tavily Search                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sneaker Resell Market Analyst                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "product_name": "Nike Book 1 'Torched'",                                                                 │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-30",                                                                            │
│        "retail_price": 155,                                                                                     │
│        "retailers": null,                                                                                       │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/"                                                          │
│        ],                                                                                                       │
│        "resale_estimate": 180,                                                                                  │
│        "confidence_score": 70                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Nike GT Future 'Lightning'",                                                            │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-31",                                                                            │
│        "retail_price": 210,                                                                                     │
│        "retailers": null,                                                                                       │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/"                                                          │
│        ],                                                                                                       │
│        "resale_estimate": 230,                                                                                  │
│        "confidence_score": 75                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Air Jordan 3 GS 'We Outside'",                                                          │
│        "brand": "Air Jordan",                                                                                   │
│        "release_date": "2026-01-01",                   

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e7b9f5a2-4c16-42ef-9437-0be4be7b497a                                                                     │
│  Agent: Sneaker Resell Market Analyst                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='{\n  "items": [\n    {\n      "product_name": "Nike Book 1 \'Torched\'",\n      "brand": "Nike",\n      "release_date": "2025-12-30",\n      "retail_price": 155,\n      "retailers": null,\n      "seed_sources": [\n        "https://www.sneakerfiles.com/release-dates/"\n      ],\n      "resale_estimate": 180,\n      "confidence_score": 70\n    },\n    {\n      "product_name": "Nike GT Future \'Lightning\'",\n      "brand": "Nike",\n      "release_date": "2025-12-31",\n      "retail_price": 210,\n      "retailers": null,\n      "seed_sources": [\n        "https://www.sneakerfiles.com/release-dates/"\n      ],\n      "resale_estimate": 230,\n      "confidence_score": 75\n    },\n    {\n      "product_name": "Air Jordan 3 GS \'We Outside\'",\n      "brand": "Air Jordan",\n      "release_date": "2026-01-01",\n      "retail_price": 155,\n      "retailers": null,\n      "seed_sources": [\n        "https://www.sneakerfiles.com/release-dates/"\n      ],\n      "resale_estimate": 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 42bb6a29-3456-4f5f-b34d-97e9b66991ac                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "product_name": "Nike Book 1 'Torched'",                                                                 │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-30",                                                                            │
│        "retail_price": 155,                                                                                     │
│        "retailers": null,                                                                                       │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/"                                                          │
│        ],                                                                                                       │
│        "resale_estimate": 180,                                                                                  │
│        "confidence_score": 70                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Nike GT Future 'Lightning'",                                                            │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-31",                                                                            │
│        "retail_price": 210,                                                                                     │
│        "retailers": null,                                                                                       │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/"                                                          │
│        ],                                                                                                       │
│        "resale_estimate": 230,                                                                                  │
│        "confidence_score": 75                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Air Jordan 3 GS 'We Outside'",                                                          │
│        "brand": "Air Jordan",                         

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯